In [21]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from sklearn.neighbors import KernelDensity
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
import numpy as np
from sklearn.mixture import GaussianMixture

In [5]:
spark = (
    SparkSession.builder
    .appName("CERNSparkViz-Synthetic")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} démarré")

Spark 4.1.1 démarré


In [6]:
df_sample = (
    spark.read
    .parquet(r"C:\Users\Tangu\cernviz\data\processed\dimuon_parquet\dimuon.parquet")
    .sample(fraction=0.01)
    .toPandas()
)

print(f"Échantillon chargé : {len(df_sample):,} paires")
print(df_sample[["pt1", "eta1", "phi1"]].describe().round(3))

Échantillon chargé : 237,390 paires
              pt1        eta1        phi1
count  237390.000  237390.000  237390.000
mean       20.862      -0.039      -0.112
std        13.810       1.315       1.824
min         3.001      -2.400      -3.142
25%        10.968      -1.197      -1.701
50%        15.958      -0.066      -0.219
75%        27.293       1.130       1.467
max        99.988       2.400       3.142


In [7]:
X_train = np.column_stack([
    np.log(df_sample["pt1"].values),   # log(pt) — plus symétrique que pt brut
    df_sample["eta1"].values,
    df_sample["phi1"].values,
])

print(f"Shape X_train : {X_train.shape}")
print(f"Variables : log(pt), eta, phi")

# Entraînement KDE
# bandwidth — contrôle le lissage. 0.1 = assez fin pour capturer les structures
kde = KernelDensity(kernel="gaussian", bandwidth=0.1)
kde.fit(X_train)

print(f"\nKDE entraînée sur {len(X_train):,} muons")

Shape X_train : (237390, 3)
Variables : log(pt), eta, phi

KDE entraînée sur 237,390 muons


## Pourquoi le KDE ?

### Contexte

On veut générer des événements synthétiques qui ressemblent
aux vraies collisions du LHC.

Le problème : on n'a pas de labels.
On ne sait pas si un événement est "un J/ψ" ou "du fond" —
on a juste des valeurs de pt, eta, phi pour chaque muon.

### Ce qu'est le KDE

Le KDE (Kernel Density Estimation) est un modèle **non supervisé**.
Il ne cherche pas à prédire une valeur — il cherche à apprendre
la **forme** de la distribution des données.

Concrètement : pour chaque point observé, on place une petite
cloche gaussienne. En additionnant toutes les cloches,
on obtient une courbe lisse qui représente la densité des données.

### L'intérêt ici

Une fois la distribution apprise, on peut **générer de nouveaux points**
qui suivent la même distribution — c'est ce qu'on appelle un
modèle génératif.

### Limite fondamentale

Le KDE apprend la distribution de chaque variable séparément.
Il ne capture pas les **corrélations** entre variables

In [12]:
N_GENERATE = 100_000

print(f"Génération de {N_GENERATE*2:,} muons synthétiques...")

# Tirer 2 fois N muons — un pour chaque côté de la paire
samples_1 = kde.sample(N_GENERATE)  # muon 1
samples_2 = kde.sample(N_GENERATE)  # muon 2

# Reconvertir log(pt) → pt
pt1_syn  = np.exp(samples_1[:, 0])
eta1_syn = samples_1[:, 1]
phi1_syn = samples_1[:, 2]

pt2_syn  = np.exp(samples_2[:, 0])
eta2_syn = samples_2[:, 1]
phi2_syn = samples_2[:, 2]

# Appliquer les mêmes filtres que sur les données réelles
mask = (
    (pt1_syn >= 3) & (pt1_syn < 100) &
    (pt2_syn >= 3) & (pt2_syn < 100) &
    (np.abs(eta1_syn) <= 2.4) &
    (np.abs(eta2_syn) <= 2.4)
)

pt1_syn, pt2_syn   = pt1_syn[mask], pt2_syn[mask]
eta1_syn, eta2_syn = eta1_syn[mask], eta2_syn[mask]
phi1_syn, phi2_syn = phi1_syn[mask], phi2_syn[mask]

print(f"Paires après filtres : {mask.sum():,}")

Génération de 200,000 muons synthétiques...
Paires après filtres : 97,427


In [9]:
mass_syn = np.sqrt(
    2 * pt1_syn * pt2_syn * (
        np.cosh(eta1_syn - eta2_syn) -
        np.cos(phi1_syn - phi2_syn)
    )
)

# Filtrer les valeurs dans la plage d'intérêt
mask_range = (mass_syn >= 0.5) & (mass_syn <= 120)
mass_syn = mass_syn[mask_range]

print(f"Paires avec masse calculée : {len(mass_syn):,}")
print(f"Masse min : {mass_syn.min():.3f} GeV")
print(f"Masse max : {mass_syn.max():.1f} GeV")
print(f"Médiane   : {np.median(mass_syn):.3f} GeV")

Paires avec masse calculée : 91,965
Masse min : 0.599 GeV
Masse max : 120.0 GeV
Médiane   : 35.242 GeV


In [10]:
N_BINS = 1000
MASS_MIN = 0.5
MASS_MAX = 120.0
BIN_WIDTH = (MASS_MAX - MASS_MIN) / N_BINS
bins = np.linspace(MASS_MIN, MASS_MAX, N_BINS + 1)
bin_centers = (bins[:-1] + bins[1:]) / 2

# Histogramme synthétique
hist_syn, _ = np.histogram(mass_syn, bins=bins)

# Histogramme réel — depuis hist_pd qu'on avait calculé avant
# Si hist_pd n'est plus en mémoire, on le recharge depuis Parquet
df_real = (
    spark.read
    .parquet(r"C:\Users\Tangu\cernviz\data\processed\dimuon_parquet\dimuon.parquet")
    .sample(fraction=0.004)  # ~100k paires pour équilibrer
    .toPandas()
)

mass_real = np.sqrt(
    2 * df_real["pt1"] * df_real["pt2"] * (
        np.cosh(df_real["eta1"] - df_real["eta2"]) -
        np.cos(df_real["phi1"] - df_real["phi2"])
    )
)
mask_real = (mass_real >= MASS_MIN) & (mass_real <= MASS_MAX)
hist_real, _ = np.histogram(mass_real[mask_real], bins=bins)

print(f"Paires réelles  : {mask_real.sum():,}")
print(f"Paires synthétiques : {len(mass_syn):,}")

Paires réelles  : 91,686
Paires synthétiques : 91,965


In [11]:
fig = go.Figure()

# Spectre réel
fig.add_trace(go.Scatter(
    x=bin_centers,
    y=hist_real,
    mode="lines",
    fill="tozeroy",
    name="Données réelles",
    line=dict(color="steelblue", width=1),
    opacity=0.7,
))

# Spectre synthétique
fig.add_trace(go.Scatter(
    x=bin_centers,
    y=hist_syn,
    mode="lines",
    fill="tozeroy",
    name="Données synthétiques (KDE)",
    line=dict(color="tomato", width=1),
    opacity=0.7,
))

fig.update_layout(
    title="Réel vs Synthétique — Spectre de masse invariante di-muon",
    xaxis=dict(
        title="Masse invariante (GeV)",
        type="log",
        tickvals=[0.5, 1, 2, 3, 5, 10, 20, 50, 91, 120],
        ticktext=["0.5", "1", "2", "3", "5", "10", "20", "50", "91", "120"],
    ),
    yaxis=dict(
        title="Nombre de paires",
        type="log",
    ),
    template="plotly_dark",
    height=550,
)

fig.show()

## Résultats — Ce que la KDE reproduit et ce qu'elle rate

### Ce qui fonctionne
Le fond continu entre 5 et 50 GeV est bien reproduit.
La forme générale du spectre est reconnaissable.

### Ce qui échoue
**Les pics de résonance ont disparu** — le J/ψ est très atténué,
le Z⁰ est quasi-absent dans le synthétique.

**Basse masse sous-représentée** — le synthétique est en dessous
du réel entre 0.5 et 5 GeV.

**Haute masse sur-représentée** — le synthétique dépasse le réel
entre 10 et 80 GeV.

### Pourquoi ?

Les résonances (J/ψ, Z⁰) ne sont pas des propriétés d'un muon individuel —
ce sont des **corrélations entre les deux muons d'une paire**.

Quand un J/ψ se désintègre en deux muons, les deux muons ne sont pas
indépendants : leur angle relatif, leurs impulsions, leurs directions
sont contraints par la masse et le spin de la particule mère.

En générant chaque muon indépendamment depuis la même KDE,
on détruit exactement cette corrélation — les deux muons
ne "savent pas" qu'ils viennent de la même désintégration.

---

## Piste 2 — Capturer les corrélations entre les deux muons

### Le problème

On entraînait la KDE sur 3 variables (pt, eta, phi d'un muon).
Il faut l'entraîner sur **6 variables simultanément** :
pt1, eta1, phi1, pt2, eta2, phi2.

### Pourquoi ça change tout

Avec une KDE 6D, chaque point d'entraînement est une **paire complète**.
La KDE apprend donc la distribution jointe des deux muons
y compris leurs corrélations.

Quand on tire un nouvel échantillon, on tire une paire entière (6 valeurs)
depuis cette distribution jointe. Les deux muons générés sont corrélés
exactement comme dans les données réelles.

### Ce qu'on attend

Les pics de résonance devraient réapparaître dans le spectre synthétique —
parce que les corrélations qui les produisent sont maintenant capturées.

In [17]:
# On entraîne sur les 6 variables simultanément
# pour capturer les corrélations entre les deux muons

# Réduire l'échantillon — KDE 6D est plus coûteux
df_kde = df_sample.sample(n=100_000, random_state=42)

X_train_6d = np.column_stack([
    np.log(df_kde["pt1"].values),   # log(pt1)
    df_kde["eta1"].values,           # eta1
    df_kde["phi1"].values,           # phi1
    np.log(df_kde["pt2"].values),   # log(pt2)
    df_kde["eta2"].values,           # eta2
    df_kde["phi2"].values,           # phi2
])

print(f"Shape X_train_6D : {X_train_6d.shape}")

kde_6d = KernelDensity(kernel="gaussian", bandwidth=0.1)
kde_6d.fit(X_train_6d)

print("KDE 6D entraînée.")

Shape X_train_6D : (100000, 6)
KDE 6D entraînée.


In [16]:
# On tire des paires complètes 
samples_6d = kde_6d.sample(100_000)

# Reconvertir log(pt) → pt
pt1_syn  = np.exp(samples_6d[:, 0])
eta1_syn = samples_6d[:, 1]
phi1_syn = samples_6d[:, 2]
pt2_syn  = np.exp(samples_6d[:, 3])
eta2_syn = samples_6d[:, 4]
phi2_syn = samples_6d[:, 5]

# Mêmes filtres que les données réelles
mask = (
    (pt1_syn >= 3) & (pt1_syn < 100) &
    (pt2_syn >= 3) & (pt2_syn < 100) &
    (np.abs(eta1_syn) <= 2.4) &
    (np.abs(eta2_syn) <= 2.4)
)

pt1_syn  = pt1_syn[mask]
pt2_syn  = pt2_syn[mask]
eta1_syn = eta1_syn[mask]
eta2_syn = eta2_syn[mask]
phi1_syn = phi1_syn[mask]
phi2_syn = phi2_syn[mask]

print(f"Paires après filtres : {mask.sum():,}")

Paires après filtres : 97,880


In [19]:
# Masse invariante KDE 6D
mass_syn_6d = np.sqrt(
    2 * pt1_syn * pt2_syn * (
        np.cosh(eta1_syn - eta2_syn) -
        np.cos(phi1_syn - phi2_syn)
    )
)

mask_range = (mass_syn_6d >= 0.5) & (mass_syn_6d <= 120)
mass_syn_6d = mass_syn_6d[mask_range]

print(f"Paires avec masse calculée : {len(mass_syn_6d):,}")
print(f"Médiane synthétique 6D : {np.median(mass_syn_6d):.3f} GeV")

Paires avec masse calculée : 96,864
Médiane synthétique 6D : 24.220 GeV


In [20]:
bins = np.linspace(0.5, 120, 1001)
bin_centers = (bins[:-1] + bins[1:]) / 2

hist_syn_6d, _ = np.histogram(mass_syn_6d, bins=bins)
hist_syn_3d, _ = np.histogram(mass_syn, bins=bins)
hist_real, _   = np.histogram(
    np.sqrt(
        2 * df_real["pt1"] * df_real["pt2"] * (
            np.cosh(df_real["eta1"] - df_real["eta2"]) -
            np.cos(df_real["phi1"] - df_real["phi2"])
        )
    ).pipe(lambda x: x[(x >= 0.5) & (x <= 120)]),
    bins=bins
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=bin_centers, y=hist_real,
    mode="lines", fill="tozeroy",
    name="Réel",
    line=dict(color="steelblue", width=1),
    opacity=0.7,
))

fig.add_trace(go.Scatter(
    x=bin_centers, y=hist_syn_3d,
    mode="lines",
    name="Synthétique 3D",
    line=dict(color="tomato", width=1.5, dash="dash"),
))

fig.add_trace(go.Scatter(
    x=bin_centers, y=hist_syn_6d,
    mode="lines",
    name="Synthétique 6D",
    line=dict(color="limegreen", width=1.5),
))

fig.update_layout(
    title="Réel vs Synthétique 3D vs Synthétique 6D",
    xaxis=dict(
        title="Masse invariante (GeV)",
        type="log",
        tickvals=[0.5, 1, 2, 3, 5, 10, 20, 50, 91, 120],
        ticktext=["0.5", "1", "2", "3", "5", "10", "20", "50", "91", "120"],
    ),
    yaxis=dict(title="Nombre de paires", type="log"),
    template="plotly_dark",
    height=550,
)

fig.show()

## Résultats — KDE 3D vs KDE 6D

### Ce qu'on observe

| Métrique | Réel | KDE 3D | KDE 6D |
|----------|------|--------|--------|
| Médiane masse | ~10 GeV | 35.2 GeV | 24.2 GeV |
| Fond 5-50 GeV | référence | sous-estimé | proche |
| Pics de résonance | oui | quasi-absents | très atténués |
| Z⁰ à 91 GeV | oui | absent | visible |

### Interprétation

La KDE 6D améliore significativement la KDE 3D.
Le fond continu est mieux reproduit et le Z⁰ commence à apparaître.
Mais les pics de résonance restent très atténués.

### Limite fondamentale du KDE

Les résonances représentent une fraction très faible des données
dans leur région de masse.
Le KDE applique le même lissage partout avec un bandwidth fixe.

**Le KDE est un modèle global. Il ne distingue pas
les zones denses des zones creuses.**

---

## Piste 3 — Gaussian Mixture Model (GMM)

### Pourquoi le GMM est une amélioration naturelle

Le KDE place une cloche gaussienne sur **chaque point** observé.
Avec 100 000 points d'entraînement, c'est 100 000 cloches.

Le GMM apprend **K gaussiennes** qui résument la distribution globale.
Chaque gaussienne peut capturer une structure différente.
Certaines peuvent naturellement se positionner sur les résonances,
d'autres sur le fond continu.

### Ce qu'on attend

Les gaussiennes du GMM devraient mieux capturer
les concentrations locales de paires autour des résonances,
là où le KDE lissait tout uniformément.

### Paramètre clé : K (nombre de gaussiennes)

Trop peu (K=2) → trop grossier, rate les structures fines.
Trop grand (K=100) → surapprentissage, lent.
On commence avec K=20 et on ajuste selon les résultats.

In [22]:
gmm = GaussianMixture(
    n_components=20,    # 20 gaussiennes
    covariance_type="full",  # chaque gaussienne a sa propre matrice de covariance
    random_state=42,
    verbose=1           # affiche la progression
)

gmm.fit(X_train_6d)

print(f"\nGMM entraîné.")
print(f"Converged : {gmm.converged_}")
print(f"Iterations : {gmm.n_iter_}")

Initialization 0
  Iteration 10
  Iteration 20
  Iteration 30
  Iteration 40
Initialization converged.

GMM entraîné.
Converged : True
Iterations : 42


In [23]:
# Génération 100k paires depuis le GMM
print("Génération de 100 000 paires synthétiques GMM...")

samples_gmm = gmm.sample(100_000)[0]

# Reconvertir log(pt) → pt
pt1_gmm  = np.exp(samples_gmm[:, 0])
eta1_gmm = samples_gmm[:, 1]
phi1_gmm = samples_gmm[:, 2]
pt2_gmm  = np.exp(samples_gmm[:, 3])
eta2_gmm = samples_gmm[:, 4]
phi2_gmm = samples_gmm[:, 5]

# Mêmes filtres
mask = (
    (pt1_gmm >= 3) & (pt1_gmm < 100) &
    (pt2_gmm >= 3) & (pt2_gmm < 100) &
    (np.abs(eta1_gmm) <= 2.4) &
    (np.abs(eta2_gmm) <= 2.4)
)

pt1_gmm  = pt1_gmm[mask]
pt2_gmm  = pt2_gmm[mask]
eta1_gmm = eta1_gmm[mask]
eta2_gmm = eta2_gmm[mask]
phi1_gmm = phi1_gmm[mask]
phi2_gmm = phi2_gmm[mask]

print(f"Paires après filtres : {mask.sum():,}")

# Masse invariante
mass_gmm = np.sqrt(
    2 * pt1_gmm * pt2_gmm * (
        np.cosh(eta1_gmm - eta2_gmm) -
        np.cos(phi1_gmm - phi2_gmm)
    )
)
mask_range = (mass_gmm >= 0.5) & (mass_gmm <= 120)
mass_gmm = mass_gmm[mask_range]

print(f"Paires avec masse calculée : {len(mass_gmm):,}")
print(f"Médiane GMM  : {np.median(mass_gmm):.3f} GeV")

Génération de 100 000 paires synthétiques GMM...
Paires après filtres : 94,096
Paires avec masse calculée : 90,708
Médiane GMM  : 23.146 GeV


c:\Users\Tangu\anaconda3\envs\cernviz\Lib\site-packages\sklearn\mixture\_base.py:468: RuntimeWarning: covariance is not symmetric positive-semidefinite.
  rng.multivariate_normal(mean, covariance, int(sample))


In [24]:
hist_gmm, _ = np.histogram(mass_gmm, bins=bins)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=bin_centers, y=hist_real,
    mode="lines", fill="tozeroy",
    name="Réel",
    line=dict(color="steelblue", width=1),
    opacity=0.6,
))

fig.add_trace(go.Scatter(
    x=bin_centers, y=hist_syn_6d,
    mode="lines",
    name="KDE 6D",
    line=dict(color="tomato", width=1.5, dash="dash"),
))

fig.add_trace(go.Scatter(
    x=bin_centers, y=hist_gmm,
    mode="lines",
    name="GMM (K=20)",
    line=dict(color="limegreen", width=1.5),
))

fig.update_layout(
    title="Réel vs KDE 6D vs GMM — Spectre de masse invariante",
    xaxis=dict(
        title="Masse invariante (GeV)",
        type="log",
        tickvals=[0.5, 1, 2, 3, 5, 10, 20, 50, 91, 120],
        ticktext=["0.5", "1", "2", "3", "5", "10", "20", "50", "91", "120"],
    ),
    yaxis=dict(title="Nombre de paires", type="log"),
    template="plotly_dark",
    height=550,
)

fig.show()

## Résultats — KDE 3D vs KDE 6D

### Ce qu'on observe

| Métrique | Réel | KDE 6D | GMM |
|----------|------|--------|--------|
| Médiane masse | ~10 GeV | 24.2 GeV | 23.146 GeV |
| Fond 5-50 GeV | référence | proche | très proche |
| Pics de résonance | oui | très atténués | presque absents |
| Z⁰ à 91 GeV | oui | visible | à peine visible |

### Interprétation

Le GMM tend à mieux représenter le fond continu mais
performe moins bien que le KDE 6D pour les pics de résonnance.
On a même une perte de performance pour Z⁰, en comparaison.

---

## Résultats

On aura réussi à recréer des évènements plutôts similaires
**au-dessus de 20 GeV**, ainsi qu'à approximer correctement le 
fond continu.
Nos algorithmes génératifs semblent néanmons **atteindre leurs limites**
quand il s'agit de recréer les pics de résonnance, notamment en-dessous de 20 GeV

## Pistes d'exploration

- La piste la plus évidente serait de **reprendre le KDE 6D et d'essayer d'apater la Bandwith**
qui nous posait problèmes, selon la densité locale, pour mieux approximer les pics de résonnance, avec plus de précision
- Entraîner le **KDE 6D sur des zone séparées** (exemple: entre 2.5 et 3.5 GeV J/ψ), mais cela introduit un **biais de connaissance**
 des résultats qui irait à l'encontre de notre démarche
- Le **GAN** (Generative Adversarial Network): un générateur et un discriminateur s'entraînent en opposition.
Le générateur essaie de tromper le discriminateur,
qui essaie de distinguer réel de synthétique.
C'est la solution qui prometterait les meilleurs résultats mais aussi la plus complexe à mettre en place.